# 🔍 Visualización de Anomalías en Runs Nuevas (Evaluación)

Este módulo interactivo permite analizar los resultados del modelo aplicado a datos no vistos (evaluación), explorando:

- Señal `CCL` suavizada.
- Score de anomalía (Isolation Forest) por tramos.
- Tramos más anómalos resaltados.
- Exportación profesional de resultados.


### 💾 Celda 2 – Carga del dataset de evaluación

In [32]:
import pandas as pd

# Cargar resultados
df_eval = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_eval_scores.csv")

# Asegurar orden correcto
df_eval = df_eval.sort_values(by=["pozo", "etapa", "DEPT"]).reset_index(drop=True)

# Verificación
print(f"Total de filas: {len(df_eval)}")
df_eval.head()


Total de filas: 13181


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,CCL_norm_max,CCL_norm_min,abs_dCCL_mean,abs_dCCL_std,abs_dCCL_max,TENS_mean,TENS_std,TENS_max,score_iso,anomaly_iso
0,2800.00,0.03120,363.11028,BPE-2341_E46_Down__27Nov24_104418.las,BPE-2341,Down,E46,3.120936,NaN,NaN,...,10.0,-9.954986,0.903328,1.097858,19.044713,815.354676,115.577624,1222.35097,-0.025534,1
1,2800.05,0.03104,364.00007,BPE-2341_E46_Down__27Nov24_104418.las,BPE-2341,Down,E46,3.104931,-0.016005,0.016005,...,10.0,-9.954986,0.903328,1.097858,19.044713,815.354676,115.577624,1222.35097,-0.036214,1
2,2800.10,0.01606,364.00007,BPE-2341_E46_Down__27Nov24_104418.las,BPE-2341,Down,E46,1.606482,-1.498450,1.498450,...,10.0,-9.954986,0.903328,1.097858,19.044713,815.354676,115.577624,1222.35097,-0.086150,1
3,2800.15,0.00351,364.00007,BPE-2341_E46_Down__27Nov24_104418.las,BPE-2341,Down,E46,0.351105,-1.255377,1.255377,...,10.0,-9.954986,0.903328,1.097858,19.044713,815.354676,115.577624,1222.35097,-0.122924,1
4,2800.20,-0.00643,364.00007,BPE-2341_E46_Down__27Nov24_104418.las,BPE-2341,Down,E46,-0.643193,-0.994298,0.994298,...,10.0,-9.954986,0.903328,1.097858,19.044713,815.354676,115.577624,1222.35097,-0.131071,1


### 🧰 Celda 3 – Herramientas interactivas

In [55]:
import plotly.graph_objs as go
import plotly.colors as colors
import ipywidgets as widgets
from IPython.display import display

# Dropdowns dinámicos
pozo_dropdown = widgets.Dropdown(options=sorted(df_eval["pozo"].unique()), description="Pozo:")
etapa_dropdown = widgets.Dropdown(description="Etapa:")

def update_etapas(pozo_sel):
    etapas = df_eval[df_eval["pozo"] == pozo_sel]["etapa"].unique()
    etapa_dropdown.options = sorted(etapas)

pozo_dropdown.observe(lambda change: update_etapas(change["new"]), names="value")
update_etapas(pozo_dropdown.value)

step_dropdown = widgets.Dropdown(options=[2.5, 5, 10, 15, 25, 50], value=50, description="Paso (m):")

display(pozo_dropdown, etapa_dropdown, step_dropdown)


Dropdown(description='Pozo:', options=('BPE-2341',), value='BPE-2341')

Dropdown(description='Etapa:', options=('E46',), value=None)

Dropdown(description='Paso (m):', index=5, options=(2.5, 5, 10, 15, 25, 50), value=50)

### 📊 Celda 4 – Función de visualización completa

In [56]:
def plot_eval_riesgo(df, pozo, etapa, step):
    df_et = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values("DEPT").copy()
    df_et["CCL_smooth"] = df_et["CCL"].rolling(window=10, min_periods=1).mean()
    bin_col = f"DEPT_bin_{step}"
    df_et[bin_col] = (df_et["DEPT"] // step) * step

    # Score promedio por tramo
    grouped = df_et.groupby(bin_col)["score_iso"].mean().reset_index().rename(columns={"score_iso": f"score_mean_{step}"})
    score_col = f"score_mean_{step}"
    top5 = grouped.sort_values(score_col, ascending=False).head(5).reset_index(drop=True)

    # Colores según severidad
    max_s = top5[score_col].max()
    min_s = top5[score_col].min()
    scale = colors.sequential.OrRd

    def s2color(score):
        idx = int(((score - min_s) / (max_s - min_s + 1e-5)) * (len(scale) - 1))
        return scale[idx]

    # Gráfico
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_et["CCL_smooth"],
        y=df_et["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=grouped[score_col],
        y=grouped[bin_col],
        mode='lines+markers',
        name=f'Score medio cada {step}m',
        line=dict(color='black', width=2),
        marker=dict(size=6)
    ))

    for _, row in top5.iterrows():
        y0 = row[bin_col]
        y1 = y0 + step
        fig.add_shape(
            type="rect", x0=0, x1=1, xref="paper",
            y0=y0, y1=y1, yref="y",
            fillcolor=s2color(row[score_col]), opacity=0.3,
            line_width=0, layer="below"
        )

    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa} | Paso: {step} m",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange="reversed",
        height=700,
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        font=dict(family="Arial", size=14)
    )

    fig.show()

    print(f"📋 Top 5 tramos más anómalos ({step} m):")
    display(top5)

    return fig, top5


### 💾 Celda 5 – Ejecutar visualización y exportar informe

In [57]:
fig, top5 = plot_eval_riesgo(df_eval, pozo_dropdown.value, etapa_dropdown.value, step_dropdown.value)

# Exportar si querés
nombre_archivo = f"reporte_eval_{pozo_dropdown.value}_etapa{etapa_dropdown.value}_{step_dropdown.value}m.html"
fig.write_html(nombre_archivo)
print(f"✅ Gráfico exportado a: {nombre_archivo}")

# Guardar tabla top5
top5.to_csv(nombre_archivo.replace(".html", "_top5.csv"), index=False)


📋 Top 5 tramos más anómalos (2.5 m):


,DEPT_bin_2.5,score_mean_2.5
0,3472.5,-0.034313
1,3447.5,-0.079029
2,3432.5,-0.084138
3,3367.5,-0.092661
4,3147.5,-0.099377


✅ Gráfico exportado a: reporte_eval_BPE-2341_etapaE46_2.5m.html
